# 03 — Recuperación RAG, baseline y ablación

Comparamos la búsqueda léxica BM25 con nuestra recuperación híbrida BM25 + multilingual-e5-base + bge-reranker-v2-m3. Elegimos los pesos solo con consultas de desarrollo y medimos una única vez sobre las consultas de evaluación.


In [1]:
%pip -q install -r requirements-lab4-modular.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements-lab4-modular.txt'
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "lab4_asistente").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "src" / "lab4_asistente").is_dir()), ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Raíz del proyecto:", ROOT)
import json
import numpy as np
import pandas as pd
import torch
from lab4_asistente import config as C
from lab4_asistente.recuperacion import Recuperador, precomputar, ajustar, evaluar, evaluar_catalogo


Raíz del proyecto: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz


## Cargar catálogo, consultas y etiquetas

In [3]:
catalogo = pd.read_parquet(C.RUTA_DATOS / "catalogo.parquet")
consultas = pd.read_parquet(C.RUTA_DATOS / "consultas.parquet")
qrels = pd.read_parquet(C.RUTA_EVAL / "qrels.parquet")
dev = consultas[consultas.conjunto == "dev"].reset_index(drop=True)
eval_final = consultas[consultas.conjunto == "eval"].reset_index(drop=True)
qrels_dev = qrels[qrels.query_id.isin(dev.query_id)]
qrels_eval = qrels[qrels.query_id.isin(eval_final.query_id)]
if not torch.cuda.is_available():
    raise RuntimeError("CUDA no está disponible. Selecciona el kernel .venv-dl-cuda.")
torch.arange(1, device="cuda")
torch.cuda.synchronize()
DEVICE = "cuda:0"
print(f"{len(catalogo):,} productos | {len(dev)} consultas dev | {len(eval_final)} consultas eval | {DEVICE}")


10,688 productos | 150 consultas dev | 250 consultas eval | cuda:0


## Construir recuperación BM25 + e5 + reranker

In [4]:
rec = Recuperador(catalogo)
rec.cargar_denso(C.MODELO_EMB, DEVICE, C.RUTA_CACHE)
rec.cargar_reranker(C.MODELO_RERANK, DEVICE)
print("Recuperador listo")


/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Recuperador listo


## Ajustar α y β en desarrollo

In [5]:
pre_dev = precomputar(rec, dev, qrels_dev, C.GANANCIAS)
parametros, tabla_ajuste = ajustar(pre_dev, C.K_NDCG, C.GRID)
print("Parámetros elegidos con dev:", parametros)
display(tabla_ajuste.sort_values("ndcg_dev", ascending=False).head(10))
rec.alpha = parametros.get("alpha", 0.7)
rec.beta = parametros.get("beta_sistema", 0.8)
C.RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
parametros_path = C.RUTA_RESULTADOS / "recuperacion_parametros.json"
parametros_path.write_text(json.dumps(parametros, indent=2), encoding="utf-8")


Parámetros elegidos con dev: {'alpha': 0.7, 'beta_sistema': 0.8, 'beta_bm25+reranker': 1.0}


,metodo,alpha,beta,ndcg_dev
19,sistema,0.7,0.8,0.759177
20,sistema,0.7,0.9,0.758676
18,sistema,0.7,0.7,0.758490
32,bm25+reranker,0.7,1.0,0.756396
21,sistema,0.7,1.0,0.756396
17,sistema,0.7,0.6,0.753993
31,bm25+reranker,0.7,0.9,0.753558
16,sistema,0.7,0.5,0.750977
30,bm25+reranker,0.7,0.8,0.749911
14,sistema,0.7,0.3,0.748407


70

## Evaluación final y ablación

Los parámetros ya están fijados. Esta celda mide BM25, el sistema completo, la variante sin reranker y la variante sin denso sobre las mismas consultas de evaluación. No se ajustan parámetros con estos resultados.

In [6]:
pre_eval = precomputar(rec, eval_final, qrels_eval, C.GANANCIAS)
variantes = ["bm25", "sistema", "hibrido", "bm25+reranker"]
filas = []
for metodo in variantes:
    beta = parametros.get(f"beta_{metodo}", rec.beta)
    tabla = evaluar(pre_eval, metodo, rec.alpha, beta, C.K_NDCG)
    filas.append({"variante": metodo, "nDCG@10 candidatos anotados": tabla.ndcg.mean()})

# Métrica complementaria usando la recuperación sobre el catálogo completo.
for metodo in variantes:
    beta = parametros.get(f"beta_{metodo}", rec.beta)
    full = evaluar_catalogo(rec, eval_final, qrels_eval, metodo, C.GANANCIAS, C.K_NDCG, beta)
    filas[variantes.index(metodo)]["nDCG@10 catálogo completo"] = full.ndcg.mean()

resultados_rec = pd.DataFrame(filas)
display(resultados_rec)
C.RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
resultados_rec.to_csv(C.RUTA_RESULTADOS / "recuperacion_eval.csv", index=False)


,variante,nDCG@10 candidatos anotados,nDCG@10 catálogo completo
0,bm25,0.692541,0.562320
1,sistema,0.765617,0.685088
2,hibrido,0.732333,0.633484
3,bm25+reranker,0.768688,0.656600


## Ejemplo de productos recuperados

In [7]:
query = eval_final.iloc[0].query
ids = rec.buscar(query, k=C.K_RESULTADOS, metodo="sistema")
display(catalogo.iloc[ids][["product_id", "titulo", "marca", "color"]])
print("Consulta:", query)


,product_id,titulo,marca,color
1776,B010ON3NHO,Salerm Cosmetics Champú Purificante - 1000 ml,Salerm,Único
785,B00684D55Y,Salerm Cosmetics Champú Equilibrium - 250 ml,Salerm,
1554,B00TBTWDX8,Salerm Cosmetics Hi Repair Champú - 250 ml,Salerm,Blanco
1448,B00O79V522,SALERM CHAMPU Equilibrante 5100ml Equilibrium,Salerm,
812,B0075CMHSE,"Salerm Cosmetics Dermocalm Champú - 250 ml, Tr...",Salerm,Transparente


Consulta: , champú sin sal salerm
